In [1]:
import requests
import json
from pyshacl import validate
from rdflib import Graph, URIRef
from typing import List, Dict


In [2]:

# --- CONFIGURATION (REPLACE WITH OFFICIAL DCAT-US 3.0 URLs) ---
# NOTE: This URL should point to the official DCAT-US 3.0 SHACL file (usually in Turtle format).
SHACL_RULES_URL = "https://raw.githubusercontent.com/DOI-DO/dcat-us/main/shacl/dcat-us_3.0_shacl_shapes.ttl" # Placeholder - Find the raw URL!

# File Path for the input DCAT-US 1.1 catalog
TARGET_DATA_FILE = "agency_dcat_us_1_1_catalog.json" 

def generate_conversion_report(data_file_path: str, shacl_url: str):
    """
    Reads a DCAT-US 1.1 JSON-LD file, validates it against the DCAT-US 3.0 SHACL rules,
    and generates a human-readable conversion guide based on the failures.
    """
    
    # 1. Load the SHACL Rules Graph (The Definitive DCAT-US 3.0 Standard)
    print("1. Fetching and loading DCAT-US 3.0 SHACL Rules...")
    shacl_graph = Graph()
    try:
        shacl_rules_response = requests.get(shacl_url)
        shacl_rules_response.raise_for_status()
        shacl_graph.parse(data=shacl_rules_response.text, format='turtle')
    except Exception as e:
        print(f"FATAL ERROR: Could not load SHACL rules from {shacl_url}. Validation aborted: {e}")
        return

    # 2. Load the Agency's DCAT-US 1.1 Data (The Data Graph)
    print(f"2. Loading Agency Data from {data_file_path} (Interpreted as JSON-LD)...")
    data_graph = Graph()
    try:
        # RDFLib's JSON-LD parser converts the 1.1 data into RDF triples using the @context
        data_graph.parse(data_file_path, format='json-ld')
    except Exception as e:
        print(f"FATAL ERROR: Could not parse JSON-LD data. Check file syntax: {e}")
        return

    # 3. Perform SHACL Validation (This is the comparison step)
    print("3. Running SHACL validation (DCAT-US 3.0 rules vs. 1.1 data)...")
    conforms, results_graph, results_text = validate(
        data_graph, 
        shacl_graph=shacl_graph,
        # RDFS inference can help catch implicit type errors
        inference='rdfs', 
        abort_on_error=False
    )

    # 4. Analyze the Validation Report to Create the Guide
    print("\n" + "="*70)
    print(f"| DCAT-US 1.1 to 3.0 Conversion Analysis: {'CONFORMANT' if conforms else 'VIOLATIONS FOUND'}")
    print("="*70)

    if conforms:
        print("🎉 Congratulations! Your catalog already meets all DCAT-US 3.0 Mandatory requirements.")
        print("Review the full SHACL report for Recommended (R) enhancements.")
    else:
        # Use SPARQL to query the results_graph for structured information
        query = """
        SELECT ?focusNode ?severity ?message ?sourceShape
        WHERE {
            ?result a sh:ValidationResult .
            ?result sh:focusNode ?focusNode .
            ?result sh:resultSeverity ?severity .
            ?result sh:resultMessage ?message .
            ?result sh:sourceShape ?sourceShape .
        }
        """
        
        # Use a dictionary to map SHACL severity URIs to friendly strings
        severity_map = {
            str(URIRef("http://www.w3.org/ns/shacl#Violation")): "🛑 Mandatory Violation",
            str(URIRef("http://www.w3.org/ns/shacl#Warning")): "⚠️ Recommended Enhancement",
            str(URIRef("http://www.w3.org/ns/shacl#Info")): "ℹ️ Optional Improvement"
        }
        
        findings: List[Dict] = []
        for row in results_graph.query(query):
            findings.append({
                'Severity': severity_map.get(str(row.severity), "UNKNOWN"),
                'Entity': str(row.focusNode),
                'Violation': str(row.message).split('on property')[0].strip(),
                'Shape': str(row.sourceShape).split('/')[-1] # Get simplified shape name
            })
            
        # Output the generated guide document (Markdown format)
        output_markdown_guide(findings)

def output_markdown_guide(findings: List[Dict]):
    """Formats the structured findings into the human-readable guide document."""
    
    print("## 📄 DCAT-US 1.1 to 3.0 Conversion Guide (Analysis Results)\n")
    print("This guide is generated by comparing your existing catalog against the official DCAT-US 3.0 SHACL rules.\n")
    
    # --- Mandatory Violations Section ---
    print("## 🛑 Mandatory Field Gaps (MUST FIX)\n")
    mandatory_findings = [f for f in findings if 'Mandatory' in f['Severity']]
    if mandatory_findings:
        print("| Status | Entity URI | Required Action (SHACL Constraint) |")
        print("| :--- | :--- | :--- |")
        for f in mandatory_findings:
            # Example: Find a missing property constraint
            action = f['Violation'].replace("does not have a value", "is MISSING").replace("does not conform", "has an INVALID value type/format")
            print(f"| **{f['Severity'].split()[0]}** | `{f['Entity'].split('/')[-1]}` | **{action}** (Source: `{f['Shape']}`) |")
    else:
        print("✅ No Mandatory (M) DCAT-US 3.0 requirements were violated. Proceed to Recommended checks.\n")

    # --- Recommended Enhancements Section ---
    print("\n## ✨ Recommended Enhancements (High Value)\n")
    recommended_findings = [f for f in findings if 'Recommended' in f['Severity']]
    if recommended_findings:
        print("These issues are typically due to new 3.0 features (like structured geospatial or governance fields) that were optional or non-existent in 1.1.\n")
        print("| Status | Entity URI | Recommendation |")
        print("| :--- | :--- | :--- |")
        for f in recommended_findings:
            action = f['Violation'].replace("has a value that is not in the value set", "has an outdated value (CV mismatch)")
            print(f"| {f['Severity'].split()[0]} | `{f['Entity'].split('/')[-1]}` | **{action}** (Check new **3.0 Controlled Vocabularies** or structures like `dcat-us:GeographicBoundingBox`). |")
    else:
        print("✅ No major Recommended (R) issues found. Excellent starting point for conversion!")

    print("\n" + "="*70)
    
# --- FINAL EXECUTION ---
if __name__ == "__main__":
    # NOTE: You must have a local file named 'agency_dcat_us_1_1_catalog.json' for this to run.
    generate_conversion_report(TARGET_DATA_FILE, SHACL_RULES_URL)

1. Fetching and loading DCAT-US 3.0 SHACL Rules...
2. Loading Agency Data from agency_dcat_us_1_1_catalog.json (Interpreted as JSON-LD)...
FATAL ERROR: Could not parse JSON-LD data. Check file syntax: [Errno 2] No such file or directory: '/home/joe/work/DCAT/agency_dcat_us_1_1_catalog.json'
